In [43]:
import pandas as pd
import os

In [44]:
def limpiar_copernicus():
    ruta_raw = "../data/raw/valencia_mar_temperatura.csv"

    df = pd.read_csv(ruta_raw)
    df["time"] = pd.to_datetime(df["time"], format= "%Y-%m-%d", errors= "coerce").dt.to_period("M")
    df = df.set_index("time")

    df_temp_agua = df.loc[:,["thetao"]].copy()
    return df_temp_agua


In [45]:
def limpiar_max_min_columns(df:pd.DataFrame, column:str):
    df.loc[:,column] = df[column].str.partition("(")[0]
    df[column] = df[column].astype(float)



def limpiar_aemet():
    ruta_raw = "../data/raw/datos_climatologicos_valencia.json"

    df = pd.read_json(ruta_raw)
    # limpiar la fecha
    df["fecha"] = pd.to_datetime(df["fecha"], format= "%Y-%m", errors= "coerce").dt.to_period("M")
    df = df.dropna(subset=["fecha"])
    df = df.set_index("fecha")
    df = df.sort_index(ascending= True)
    #limpiar ta_max
    limpiar_max_min_columns(df, "ta_max")
    #limpiar ta_min
    limpiar_max_min_columns(df, "ta_min")
    #limpiar p_max
    limpiar_max_min_columns(df, "p_max")

    df_aemet =  df.loc[:,["tm_mes", "tm_max", "tm_min", "ta_max", "ta_min", "nt_30", "p_mes", "p_max", "np_100", "np_300", "n_llu", "n_gra", "n_tor"]].copy()
    return df_aemet

In [46]:
df_aemet = limpiar_aemet()
print(df_aemet.info())
df_copernicus = limpiar_copernicus()
print(df_copernicus.info())

<class 'pandas.DataFrame'>
PeriodIndex: 420 entries, 1991-01 to 2025-12
Freq: M
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   tm_mes  420 non-null    float64
 1   tm_max  420 non-null    float64
 2   tm_min  420 non-null    float64
 3   ta_max  420 non-null    float64
 4   ta_min  420 non-null    float64
 5   nt_30   420 non-null    int64  
 6   p_mes   420 non-null    float64
 7   p_max   419 non-null    float64
 8   np_100  418 non-null    float64
 9   np_300  418 non-null    float64
 10  n_llu   420 non-null    int64  
 11  n_gra   420 non-null    int64  
 12  n_tor   420 non-null    int64  
dtypes: float64(9), int64(4)
memory usage: 45.9 KB
None
<class 'pandas.DataFrame'>
PeriodIndex: 420 entries, 1991-01 to 2025-12
Freq: M
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   thetao  420 non-null    float64
dtypes: float64(1)
memory usage: 6.6 KB
None


In [47]:
filas_con_nulos = df_aemet[df_aemet.isna().any(axis=1)]
filas_con_nulos

,tm_mes,tm_max,tm_min,ta_max,ta_min,nt_30,p_mes,p_max,np_100,np_300,n_llu,n_gra,n_tor
fecha,,,,,,,,,,,,,
2005-09,23.1,27.6,18.6,37.5,14.0,5,49.0,26.2,NaN,NaN,8,0,0
2006-01,10.5,14.4,6.6,20.1,-0.6,0,66.6,NaN,NaN,NaN,7,0,0


In [59]:
p_mes_missing1 = filas_con_nulos.loc["2005-09", "p_mes"]
n_llu_missing1 = filas_con_nulos.loc["2005-09", "n_llu"]

df_aemet[(df_aemet.index.month == 9) & (df_aemet.index.year != 2005) &
         (df_aemet["p_mes"].between(p_mes_missing1 - (p_mes_missing1 * 0.1), p_mes_missing1 + (p_mes_missing1 * 0.1)))
         & (df_aemet["n_llu"].between(n_llu_missing1 - (n_llu_missing1 * 0.1), n_llu_missing1 + (n_llu_missing1 * 0.1)))]

p_mes_missing2 = filas_con_nulos.loc["2006-01", "p_mes"]
n_llu_missing2 = filas_con_nulos.loc["2006-01", "n_llu"]

df_aemet[(df_aemet.index.month == 9) & (df_aemet.index.year != 2005) &
         (df_aemet["p_mes"].between(p_mes_missing2 - (p_mes_missing2 * 0.2), p_mes_missing2 + (p_mes_missing2 * 0.2)))
         & (df_aemet["n_llu"].between(n_llu_missing2 - (n_llu_missing2 * 0.2), n_llu_missing2 + (n_llu_missing2 * 0.2)))]



,tm_mes,tm_max,tm_min,ta_max,ta_min,nt_30,p_mes,p_max,np_100,np_300,n_llu,n_gra,n_tor
fecha,,,,,,,,,,,,,
1999-09,24.5,29.9,19.1,38.4,15.4,13,59.0,24.6,2.0,0.0,7,1,5
2014-09,25.5,29.3,21.5,36.7,17.5,17,54.8,25.5,2.0,0.0,8,0,3


In [42]:
df = pd.concat([df_aemet, df_copernicus], axis=1)

df = df.rename(columns={"tm_mes" : "temp_med_mes", "tm_max" : "temp_med_max", "tm_min" : "temp_med_min", "ta_max" : "temp_abs_max", 
                        "ta_min" : "temp_abs_min", "nt_30" : "dias_temp>30", "p_mes" : "total_precip_mes", "p_max" : "precip_max", "np_100" : "dias_precip>10mm",
                        "np_300" : "dias_precip>30mm", "n_llu" : "dias_lluvia", "n_gra" : "dias_granizo", "n_tor" : "dias_tormenta", "thetao" : "temp_med_mar"})

df = df[["temp_med_mar", "temp_med_mes", "temp_med_max", "temp_med_min", "temp_abs_max", "temp_abs_min", "dias_temp>30", 
         "total_precip_mes", "precip_max", "dias_precip>10mm", "dias_precip>30mm", "dias_lluvia", "dias_granizo", "dias_tormenta"]]

df.to_parquet("../data/processed/datos_meteo_Valencia_90-25.parquet")

print(df.info())

display(df)

<class 'pandas.DataFrame'>
PeriodIndex: 420 entries, 1991-01 to 2025-12
Freq: M
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   temp_med_mar      419 non-null    float64
 1   temp_med_mes      420 non-null    float64
 2   temp_med_max      420 non-null    float64
 3   temp_med_min      420 non-null    float64
 4   temp_abs_max      420 non-null    float64
 5   temp_abs_min      420 non-null    float64
 6   dias_temp>30      420 non-null    int64  
 7   total_precip_mes  420 non-null    float64
 8   precip_max        419 non-null    float64
 9   dias_precip>10mm  418 non-null    float64
 10  dias_precip>30mm  418 non-null    float64
 11  dias_lluvia       420 non-null    int64  
 12  dias_granizo      420 non-null    int64  
 13  dias_tormenta     420 non-null    int64  
dtypes: float64(10), int64(4)
memory usage: 49.2 KB
None


,temp_med_mar,temp_med_mes,temp_med_max,temp_med_min,temp_abs_max,temp_abs_min,dias_temp>30,total_precip_mes,precip_max,dias_precip>10mm,dias_precip>30mm,dias_lluvia,dias_granizo,dias_tormenta
1991-01,13.602020,11.1,15.3,6.8,22.5,1.2,0,104.2,32.4,3.0,2.0,7,1,1
1991-02,12.535235,10.9,15.1,6.6,21.5,0.4,0,67.7,29.5,2.0,0.0,7,0,1
1991-03,13.396119,14.5,18.9,10.1,26.6,6.0,0,82.7,46.0,2.0,1.0,15,0,0
1991-04,14.474826,14.7,19.5,9.8,24.5,5.6,0,37.5,14.9,2.0,0.0,10,0,1
1991-05,16.517714,16.2,21.3,11.0,27.5,6.0,0,21.3,5.4,0.0,0.0,8,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08,27.919468,28.3,32.3,24.2,38.2,22.5,30,2.4,2.4,0.0,0.0,3,0,1
2025-09,25.750820,24.9,28.7,21.0,32.8,16.7,11,121.5,96.8,2.0,1.0,6,0,3
2025-10,22.869558,21.3,25.2,17.3,30.0,11.2,2,178.3,65.6,5.0,2.0,10,0,3
2025-11,19.383938,16.3,21.0,11.6,27.8,6.2,0,9.9,6.8,0.0,0.0,7,0,1
